#Ingresar los datos en formato data frame con columnas:
- X (parámetro escalado entre 0 y 1) "X_esc"
- Y (parámetro escalado entre 0 y 1) "Y_esc"
- Kt "Kt"
- Sen(Hmax) "senHMAX"
- Transmitancia calculada con observaciones "transmitancia"
- Error heterocedástico (por observación) "error_T"
- PPFD dentro calculada "PAR"
- PPFD fuera observada "PAR_OUT"

Un data frame por tratamiento

#Bootstrap:
1) Toma sub-muestra de los datos observados
2) Aparta datos no muestreados
3) Entrena modelo y predice con la sub-muestra de (1)
4) Calcula estadisticos entre modelado en (3) y datos de (2) (out of bag)
LOOP DE 1 A 4 de R repeticiones
5) Obtengo distribución de estadisticos y parámetros

- Modelo 1: transmitancia constante en tiempo y espacio t = cte
- Modelo 2: transmitancia constante en el tiempo y variable en espacio t = t(X,Y) (modelado con GPR kernel RBF + alpha)
- Modelo 3: transmitancia variable en tiempo y espacio t = t(X,Y,Kt,senHMAX) (modelado con GPR cuadratico selectivo + RBF + White noise + alpha)

In [ ]:
import numpy as np
import pandas as pd
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Kernel, Hyperparameter, RBF, WhiteKernel
from sklearn.utils.validation import check_array
from joblib import Parallel, delayed

# =========================================================================
# NOMBRES DE LOS 14 TÉRMINOS (Para rotular las columnas de la pestaña 2)
# =========================================================================
NOMBRES_TERMINOS = [
    "X_esc", "Y_esc", "Kt", "senHMAX",
    "X_esc_cuad", "Y_esc_cuad", "Kt_cuad", "senHMAX_cuad",
    "X_Y", "X_Kt", "X_senHMAX", "Y_Kt", "Y_senHMAX", "Kt_senHMAX"
]

# =========================================================================
# 1) TU KERNEL CUADRÁTICO SELECTIVO (Modelo 3)
# =========================================================================
class QuadraticSelectiveKernel(Kernel):
    def __init__(self, active_terms=None, theta_bounds=(1e-5, 1e5)):
        super().__init__()
        if active_terms is None:
            self.active_terms = np.ones(14, dtype=bool)
        else:
            self.active_terms = np.asarray(active_terms, dtype=bool)

        if self.active_terms.size != 14: raise ValueError("active_terms debe tener longitud 14")
        self.n_active = self.active_terms.sum()
        self.theta_bounds = theta_bounds
        self._theta_val = np.ones(self.n_active)
        self.theta_hyperparameter = Hyperparameter("theta", "numeric", (np.log(theta_bounds[0]), np.log(theta_bounds[1])), self.n_active)

    @property
    def theta(self): return np.log(self._theta_val)
    @theta.setter
    def theta(self, value): self._theta_val = np.exp(value)
    @property
    def bounds(self): return self.theta_hyperparameter.bounds

    def _phi_full(self, X):
        X = check_array(X)
        if X.shape[1] != 4: raise ValueError("Se esperaban 4 predictores")
        x1, x2, x3, x4 = X.T
        return np.column_stack([x1, x2, x3, x4, x1**2, x2**2, x3**2, x4**2, x1*x2, x1*x3, x1*x4, x2*x3, x2*x4, x3*x4])

    def _phi(self, X): return self._phi_full(X)[:, self.active_terms]

    def __call__(self, X, Y=None, eval_gradient=False):
        phi_X = self._phi(X)
        phi_Y = phi_X if Y is None else self._phi(Y)
        K = phi_X @ np.diag(self._theta_val) @ phi_Y.T
        if eval_gradient:
            grads = [self._theta_val[i] * np.outer(phi_X[:, i], phi_Y[:, i]) for i in range(self.n_active)]
            return K, np.stack(grads, axis=2)
        return K

    def diag(self, X): return np.sum(self._phi(X)**2 * self._theta_val, axis=1)
    def is_stationary(self): return False

# =========================================================================
# 2) DEFINICIÓN DE CLASE PARA MODELO 1 (Constante)
# =========================================================================
class ModeloConstante:
    def __init__(self, t):
        self.t = t
    def predict(self, X, return_std=False):
        n = X.shape[0]
        y_pred = np.full(n, self.t)
        if return_std:
            return y_pred, np.zeros(n)
        return y_pred

def fit_model_constante(df_boot):
    t_const = np.nanmean(df_boot["transmitancia"].values)
    return ModeloConstante(t_const)

# =========================================================================
# 3) ENTRENADORES GPR (Modelos 2 y 3)
# =========================================================================
def entrenar_gpr_rbf_cuadratico_selectivo(X, y, alpha, active_terms, senH_bound_lower):
    bounds = [(1e-1, 1e1), (1e-1, 1e1), (1e-2, 1e2), (senH_bound_lower, 1e2)]
    kernel = (RBF(length_scale=np.ones(X.shape[1]), length_scale_bounds=bounds) +
              QuadraticSelectiveKernel(active_terms=active_terms) +
              WhiteKernel(noise_level=1e-1, noise_level_bounds=(1e-4, 1e2)))
    gpr = GaussianProcessRegressor(kernel=kernel, alpha=alpha, normalize_y=True, n_restarts_optimizer=5, random_state=0)
    gpr.fit(X, y)
    return gpr

def fit_model_gpr_bivariado(df_boot):
    X_train = df_boot[["X_esc", "Y_esc"]].values
    y_train = df_boot["transmitancia"].values
    noise_alpha = df_boot["alpha"].values

    kernel = (RBF(length_scale=[1.0, 1.0],
                  length_scale_bounds=[(1e-3, 7/14), (1e-3, 4/6)]) +
                # length_scale_bounds = [(1e-3, 1e3, (1e-3, 1e3)]) +
              WhiteKernel(noise_level=0.1, noise_level_bounds=(1e-5, 1e1)))
    gpr = GaussianProcessRegressor(kernel=kernel, alpha=noise_alpha, n_restarts_optimizer=5, normalize_y=True, random_state=42)
    gpr.fit(X_train, y_train)
    return gpr

def fit_model_gpr_completo_noMS(df_boot):
    active_terms = [0, 1, 0, 0, 1, 1, 0, 1, 1, 1, 0, 1, 1, 0]
    X = df_boot[["X_esc", "Y_esc", "Kt", "senHMAX"]].values
    y = df_boot["transmitancia"].values
    alpha = df_boot["error_T"].values
    return entrenar_gpr_rbf_cuadratico_selectivo(X, y, alpha, active_terms, senH_bound_lower=0.13)

def fit_model_gpr_completo_MS(df_boot):
    active_terms = [0, 1, 0, 1, 1, 1, 0, 1, 1, 0, 1, 1, 0, 0]
    X = df_boot[["X_esc", "Y_esc", "Kt", "senHMAX"]].values
    y = df_boot["transmitancia"].values
    alpha = df_boot["error_T"].values
    return entrenar_gpr_rbf_cuadratico_selectivo(X, y, alpha, active_terms, senH_bound_lower=0.1)

# =========================================================================
# 4) ESTADÍSTICOS MASTER Y EXTRACTORES DE PARÁMETROS
# =========================================================================
def stats_function_master(y_obs, y_pred):
    mask = np.isfinite(y_obs) & np.isfinite(y_pred)
    y_obs = y_obs[mask]
    y_pred = y_pred[mask]

    if len(y_obs) < 5: return None

    bias = np.mean(y_pred - y_obs)
    MSD = np.mean((y_pred - y_obs)**2)
    media_obs = np.mean(y_obs)
    SB = (np.mean(y_pred) - media_obs)**2

    if np.std(y_pred) < 1e-10:
        m, r2, NU, LC = np.nan, np.nan, np.nan, np.nan
    else:
        try:
            coef = np.polyfit(y_pred, y_obs, 1)
            m, b = coef
            y_hat = b + m * y_pred
            SSE = np.sum((y_obs - y_hat)**2)
            SST = np.sum((y_obs - media_obs)**2)
            r2 = 1 - SSE / SST
            LC = (1 - r2) * np.var(y_obs)
            NU = (1 - m)**2 * np.var(y_pred)
        except Exception:
            m, r2, NU, LC = np.nan, np.nan, np.nan, np.nan

    return np.array([bias, m, r2, MSD, SB, NU, LC])

def get_rbf_white_params(gpr):
    ls_x, ls_y, ls_kt, ls_senh = np.nan, np.nan, np.nan, np.nan
    noise_level = np.nan

    if hasattr(gpr, "kernel_"):
        params = gpr.kernel_.get_params()
        for key, val in params.items():
            if key.endswith("length_scale") and not key.endswith("bounds"):
                if isinstance(val, (list, np.ndarray)):
                    if len(val) >= 2:
                        ls_x, ls_y = val[0], val[1]
                    if len(val) == 4:
                        ls_kt, ls_senh = val[2], val[3]
                else:
                    ls_x = val
            elif key.endswith("noise_level") and not key.endswith("bounds"):
                noise_level = val

    return [ls_x, ls_y, ls_kt, ls_senh, noise_level]

def get_quadratic_thetas(gpr):
    if hasattr(gpr, "kernel_"):
        for key, val in gpr.kernel_.get_params().items():
            if isinstance(val, QuadraticSelectiveKernel):
                return val._theta_val
    return np.full(8, np.nan) # Asume que hay 8 términos activos por diseño

# =========================================================================
# 5) EVALUACIÓN EN PARALELO MULTINÚCLEO (TRANSMITANCIA + PAR + HYPERS + THETAS)
# =========================================================================
def evaluar_una_iteracion_bootstrap(data, fit_model_func, predictores, min_oob, seed, es_gpr, es_mod3=False):
    rng = np.random.default_rng(seed)
    n = len(data)

    idx_boot = rng.choice(n, size=n, replace=True)
    idx_oob  = np.setdiff1d(np.arange(n), idx_boot)

    if len(idx_oob) < min_oob: return None

    df_boot = data.iloc[idx_boot]
    df_oob  = data.iloc[idx_oob]

    try:
        model = fit_model_func(df_boot)
        X_oob = df_oob[predictores].values

        # --- VALIDACIÓN 1: Transmitancia ---
        y_trans_obs = df_oob["transmitancia"].values
        y_trans_pred, _ = model.predict(X_oob, return_std=True)
        stats_trans = stats_function_master(y_trans_obs, y_trans_pred)
        if stats_trans is None: return None

        # --- VALIDACIÓN 2: PAR Dentro Vinculado ---
        y_par_obs = df_oob["PAR"].values
        y_par_pred = y_trans_pred * df_oob["PAR_OUT"].values
        stats_par = stats_function_master(y_par_obs, y_par_pred)
        if stats_par is None: return None

        # --- EXTRACCIÓN DE PARÁMETROS DEL KERNEL RBF/WHITE ---
        if es_gpr:
            log_lik = model.log_marginal_likelihood_value_
            hypers = get_rbf_white_params(model)
            lml_block = [log_lik] + hypers # LML + 5 Hypers
        else:
            lml_block = [np.nan, np.nan, np.nan, np.nan, np.nan, np.nan]

        resultado = np.concatenate([stats_trans, stats_par, lml_block])

        # --- EXTRACCIÓN EXTRA SI ES MODELO 3 (Los 8 Thetas) ---
        if es_mod3:
            thetas = get_quadratic_thetas(model)
            resultado = np.concatenate([resultado, thetas])

        return resultado

    except Exception:
        return None

def ejecutar_bootstrap_master_paralelo(data, fit_model_func, predictores, R=100, random_state=123, min_oob=10, es_gpr=True, es_mod3=False):
    rng = np.random.default_rng(random_state)
    seeds = rng.integers(0, 1000000, size=R)

    resultados_lista = Parallel(n_jobs=-1, verbose=10)(
        delayed(evaluar_una_iteracion_bootstrap)(data, fit_model_func, predictores, min_oob, seeds[i], es_gpr, es_mod3)
        for i in range(R)
    )
    return np.array([r for r in resultados_lista if r is not None])

# =========================================================================
# 6) COMPILACIÓN DE LA MEGA TABLA MULTI-PESTAÑA
# =========================================================================
def generar_tabla_completa(df_noms, df_ms, R_iter=100):
    preds_completos = ["X_esc", "Y_esc", "Kt", "senHMAX"]

    print("1/6) Corriendo Modelo 1 (Constante) - sin MS...")
    m1_noms = ejecutar_bootstrap_master_paralelo(df_noms, fit_model_constante, preds_completos, R=R_iter, es_gpr=False)
    print("2/6) Corriendo Modelo 1 (Constante) - con MS...")
    m1_ms   = ejecutar_bootstrap_master_paralelo(df_ms, fit_model_constante, preds_completos, R=R_iter, es_gpr=False)

    print("3/6) Corriendo Modelo 2 (GPR Espacial) - sin MS...")
    m2_noms = ejecutar_bootstrap_master_paralelo(df_noms, fit_model_gpr_bivariado, ["X_esc", "Y_esc"], R=R_iter, es_gpr=True)
    print("4/6) Corriendo Modelo 2 (GPR Espacial) - con MS...")
    m2_ms   = ejecutar_bootstrap_master_paralelo(df_ms, fit_model_gpr_bivariado, ["X_esc", "Y_esc"], R=R_iter, es_gpr=True)

    print("5/6) Corriendo Modelo 3 (GPR Mixto Selectivo) - sin MS... (Extrayendo Thetas)")
    m3_noms = ejecutar_bootstrap_master_paralelo(df_noms, fit_model_gpr_completo_noMS, preds_completos, R=R_iter, es_gpr=True, es_mod3=True)
    print("6/6) Corriendo Modelo 3 (GPR Mixto Selectivo) - con MS... (Extrayendo Thetas)")
    m3_ms   = ejecutar_bootstrap_master_paralelo(df_ms, fit_model_gpr_completo_MS, preds_completos, R=R_iter, es_gpr=True, es_mod3=True)

    # --- PESTAÑA 1: LA TABLA MAESTRA (Transmitancia, PAR, RBF y Ruido Blanco) ---
    variables_tabla = [
        "Transmitancia sin MS (Mod 1)", "Transmitancia sin MS (Mod 2)", "Transmitancia sin MS (Mod 3)",
        "Transmitancia con MS (Mod 1)", "Transmitancia con MS (Mod 2)", "Transmitancia con MS (Mod 3)",
        "PAR sin MS (Mod 1)", "PAR sin MS (Mod 2)", "PAR sin MS (Mod 3)",
        "PAR con MS (Mod 1)", "PAR con MS (Mod 2)", "PAR con MS (Mod 3)"
    ]
    percentiles = [25, 50, 75]

    filas_metricas = [(v, f"P{p}") for v in variables_tabla for p in percentiles]
    index_metricas = pd.MultiIndex.from_tuples(filas_metricas, names=["Variable", "Percentil"])

    columnas_metricas = ["Bias", "Pendiente", "R²", "RMSD", "MSD", "SB", "NU", "LC", "Max LML",
                         "LS_X", "LS_Y", "LS_Kt", "LS_senH", "Ruido_Blanco"]
    super_tabla = pd.DataFrame(index=index_metricas, columns=columnas_metricas)

    mapeo_bloques = {
        ("Transmitancia sin MS (Mod 1)", 0): m1_noms, ("Transmitancia sin MS (Mod 2)", 0): m2_noms, ("Transmitancia sin MS (Mod 3)", 0): m3_noms,
        ("Transmitancia con MS (Mod 1)", 0): m1_ms,   ("Transmitancia con MS (Mod 2)", 0): m2_ms,   ("Transmitancia con MS (Mod 3)", 0): m3_ms,
        ("PAR sin MS (Mod 1)", 7): m1_noms,           ("PAR sin MS (Mod 2)", 7): m2_noms,           ("PAR sin MS (Mod 3)", 7): m3_noms,
        ("PAR con MS (Mod 1)", 7): m1_ms,             ("PAR con MS (Mod 2)", 7): m2_ms,             ("PAR con MS (Mod 3)", 7): m3_ms
    }

    for (var_name, offset), res_matriz in mapeo_bloques.items():
        if len(res_matriz) == 0: continue
        for p in percentiles:
            def check_val(val, format_as_string=False):
                if np.isnan(val): return "-"
                return f"{val:.5f}" if format_as_string else val

            bias_p = np.percentile(res_matriz[:, offset + 0], p)
            m_p    = np.percentile(res_matriz[:, offset + 1], p)
            r2_p   = np.percentile(res_matriz[:, offset + 2], p)
            msd_p  = np.percentile(res_matriz[:, offset + 3], p)
            rmsd_p = np.sqrt(msd_p)
            sb_p   = np.percentile(res_matriz[:, offset + 4], p)
            nu_p   = np.percentile(res_matriz[:, offset + 5], p)
            lc_p   = np.percentile(res_matriz[:, offset + 6], p)

            # Extraemos la parte del LML y los Hyperparámetros RBF (Índices 14 a 19)
            lik_p    = np.percentile(res_matriz[:, 14], p)
            ls_x_p   = np.percentile(res_matriz[:, 15], p)
            ls_y_p   = np.percentile(res_matriz[:, 16], p)
            ls_kt_p  = np.percentile(res_matriz[:, 17], p)
            ls_sen_p = np.percentile(res_matriz[:, 18], p)
            noise_p  = np.percentile(res_matriz[:, 19], p)

            super_tabla.loc[(var_name, f"P{p}"), :] = [
                check_val(bias_p), check_val(m_p), check_val(r2_p), rmsd_p, msd_p, sb_p,
                check_val(nu_p), check_val(lc_p), check_val(lik_p),
                check_val(ls_x_p), check_val(ls_y_p), check_val(ls_kt_p), check_val(ls_sen_p), check_val(noise_p)
            ]

    # --- PESTAÑA 2: LOS THETAS AISLADOS DEL MODELO 3 ---
    filas_thetas = [(v, f"P{p}") for v in ["Modelo 3 - Sin MS", "Modelo 3 - Con MS"] for p in percentiles]
    index_thetas = pd.MultiIndex.from_tuples(filas_thetas, names=["Tratamiento", "Percentil"])
    tabla_thetas = pd.DataFrame(index=index_thetas, columns=NOMBRES_TERMINOS)

    activos_noms = np.array([0, 1, 0, 0, 1, 1, 0, 1, 1, 1, 0, 1, 1, 0], dtype=bool)
    activos_ms   = np.array([0, 1, 0, 1, 1, 1, 0, 1, 1, 0, 1, 1, 0, 0], dtype=bool)

    # Las columnas a partir de la 20 contienen los 8 Thetas para m3_noms y m3_ms
    for p in percentiles:
        if len(m3_noms) > 0:
            percentiles_theta_noms = np.percentile(m3_noms[:, 20:], p, axis=0)
            fila_noms = ["-"] * 14
            idx_theta = 0
            for i, activo in enumerate(activos_noms):
                if activo:
                    fila_noms[i] = f"{percentiles_theta_noms[idx_theta]:.5f}"
                    idx_theta += 1
            tabla_thetas.loc[("Modelo 3 - Sin MS", f"P{p}"), :] = fila_noms

        if len(m3_ms) > 0:
            percentiles_theta_ms = np.percentile(m3_ms[:, 20:], p, axis=0)
            fila_ms = ["-"] * 14
            idx_theta = 0
            for i, activo in enumerate(activos_ms):
                if activo:
                    fila_ms[i] = f"{percentiles_theta_ms[idx_theta]:.5f}"
                    idx_theta += 1
            tabla_thetas.loc[("Modelo 3 - Con MS", f"P{p}"), :] = fila_ms

    # --- GUARDADO MULTI-PESTAÑA EN DRIVE ---
    ruta_drive = "/content/drive/MyDrive/tabla_bootstrap_PARyt_ef_bounds.xlsx"

    try:
        with pd.ExcelWriter(ruta_drive, engine='xlsxwriter') as writer:
            super_tabla.to_excel(writer, sheet_name='Metricas_y_RBF')
            tabla_thetas.to_excel(writer, sheet_name='Thetas_Modelo3')
        print(f"\n¡LISTO! Súper tabla guardada con 2 pestañas en Drive:\n{ruta_drive}")
    except Exception as e:
        with pd.ExcelWriter("tabla_bootstrap_PARyt_ef_bounds.xlsx", engine='xlsxwriter') as writer:
            super_tabla.to_excel(writer, sheet_name='Metricas_y_RBF')
            tabla_thetas.to_excel(writer, sheet_name='Thetas_Modelo3')
        print("\n¡Aviso! Guardado en la carpeta local de Colab con dos pestañas.")

    return super_tabla

In [ ]:
# ======================================================
# 8) EJECUCIÓN DEFINITIVA DE LA TESIS
# ======================================================

# 1. Definimos la cantidad de réplicas oficiales para el Bootstrap (ej: 100)
REPLICAS_BOOTSTRAP = 100

# 2. Corremos el pipeline completo (va a imprimir el progreso de cada modelo)
# Atajamos las dos tablas que ahora escupe la función
tabla_metricas, tabla_thetas = generar_tabla_completa(
    df_noms=df_noMS,
    df_ms=df_MS,
    R_iter=REPLICAS_BOOTSTRAP
)

# 3. Visualizamos las tablas estructuradas directamente en Colab
print("\n" + "="*60)
print("--- TABLA 1: MÉTRICAS GENERALES E HIPERPARÁMETROS RBF ---")
print("="*60)
display(tabla_metricas)

print("\n" + "="*60)
print("--- TABLA 2: COEFICIENTES (THETAS) DEL MODELO 3 ---")
print("="*60)
display(tabla_thetas)